# TGFSR Equidistribution Analysis

Twisted Generalized Feedback Shift Register generators.
We test two generators with different state sizes (k=96 and k=155).

In [ ]:
from regpoly.generateur import Generateur
from regpoly.combinaison import Combinaison
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_MATRICIAL, METHOD_DUALLATTICE
)

INT_MAX = 2**31 - 1

## 1. Create TGFSR Generators

TGFSR parameters: `w` (word size), `r` (number of words), `m` (middle distance), `a` (twist mask).
The state size is $k = w \times r$.

In [ ]:
# TGFSR with k = 32*3 = 96
gen96 = Generateur.create("tgfsr", {
    "w": 32, "r": 3, "m": 1, "a": 0x9965523b
}, 32))

# TGFSR with k = 31*5 = 155
gen155 = Generateur.create("tgfsr", {
    "w": 31, "r": 5, "m": 1, "a": 0xf45c111a
}, 31))

for g in [gen96, gen155]:
    print(f"{g.name()}: k={g.k}, L={g.L}")
    g.display()

## 2. Equidistribution of Each Generator

In [ ]:
for g in [gen96, gen155]:
    comb = Combinaison(J=1, Lmax=g.L)
    comb.components[0].add_gen(g)
    comb.reset()

    test = EquidistributionTest(L=g.L, delta=[INT_MAX]*(g.L+1),
                                mse=INT_MAX, method=METHOD_MATRICIAL)
    result = test.run(comb)

    print(f"\n=== {g.name()} k={g.k} ===")
    print(f"Dimension gaps sum = {result.se}")
    result.display_table(comb, 'l')

## 3. Combined Generator (k=96 + k=155 = k=251)

XOR combination of the two generators. The combined generator has
better equidistribution than either component alone.

In [ ]:
L_comb = min(gen96.L, gen155.L)

comb2 = Combinaison(J=2, Lmax=L_comb)
comb2.components[0].add_gen(gen96)
comb2.components[1].add_gen(gen155)
comb2.reset()

print(f"Combined: k_g={comb2.k_g}, L={comb2.L}")

test = EquidistributionTest(L=comb2.L, delta=[INT_MAX]*(comb2.L+1),
                            mse=INT_MAX, method=METHOD_MATRICIAL)
result = test.run(comb2)

print(f"Dimension gaps sum = {result.se}")
result.display_table(comb2, 'l')